# Lambda Re-evaluation V2: VALIDATION + TEST Metrics

## 🎯 Purpose
Re-evaluate different lambda values for regularization and save **both VALIDATION and TEST performance**.

## 📊 V2 Changes
- ✅ Saves VALIDATION MSE/RMSE/Precision/Recall (alpha selection basis)
- ✅ Saves TEST RMSE/MAD (generalization evaluation)
- ✅ Saves regularization penalty and regularized score
- ✅ Output files marked with `_v2_` suffix

## 🔍 Key Logic
1. Load existing `alpha_optimization_history.csv` (contains VALIDATION performance)
2. For each lambda, recalculate penalty and select best alpha based on `MSE + λ×|α-1|`
3. Evaluate selected alpha on TEST data
4. Save BOTH validation metrics (from history) AND test metrics (newly computed)

## Configuration

In [1]:
import os
import pandas as pd
import numpy as np
import time
from datetime import datetime

# ==================================================================================
# CONFIGURATION
# ==================================================================================
LAMBDA_VALUES = [0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
FOLDS_TO_PROCESS = [6, 7]  # Updated to process folds 06 and 07
RELEVANCE_THRESHOLD = 4.0
DATA_DIR = "../data/movielenz_data"
RESULTS_DIR = "../results"

print(f"========================================================")
print(f"MULTI-LAMBDA SMART RE-EVALUATION V2 (WITH VALIDATION)")
print(f"Target Lambdas: {LAMBDA_VALUES}")
print(f"Target Folds: {FOLDS_TO_PROCESS}")
print(f"V2 Changes: Now saves VALIDATION + TEST performance")
print(f"========================================================\n")

MULTI-LAMBDA SMART RE-EVALUATION V2 (WITH VALIDATION)
Target Lambdas: [0.001, 0.002, 0.003, 0.004, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
Target Folds: [6, 7]
V2 Changes: Now saves VALIDATION + TEST performance



## Helper Functions

In [2]:
def _load_matrix_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    lower = [str(c).lower() for c in df.columns]
    if {"item", "user", "rating"}.issubset(lower):
        def col(name):
            for c in df.columns:
                if str(c).lower() == name: return c
            raise KeyError(name)
        i, u, r = col("item"), col("user"), col("rating")
        mat = df.pivot(index=i, columns=u, values=r)
        mat = mat.apply(pd.to_numeric, errors="coerce")
        mat.index = mat.index.astype(int)
        mat.columns = mat.columns.astype(int)
        mat = mat.sort_index().sort_index(axis=1)
        return mat

    first_col = df.columns[0]
    if str(first_col).lower() in ("item", "items", "item_id", "id", "index", "unnamed: 0"):
        df = df.set_index(first_col)
    df.index.name = None
    df.columns.name = None
    df = df.apply(pd.to_numeric, errors="coerce")
    df.index = df.index.astype(int)
    df.columns = df.columns.astype(int)
    df = df.sort_index().sort_index(axis=1)
    return df

def _combine_single_similarity(S, alpha, clip_neg=True):
    S = np.asarray(S, dtype=float)
    S_eff = np.clip(S, 0.0, None) if clip_neg else S
    S_eff = np.power(S_eff, float(alpha))
    np.fill_diagonal(S_eff, 0.0)
    return S_eff

def rmse_mad_on_test(pred_df, test_df):
    P = pred_df.values
    T = test_df.values
    mask = ~np.isnan(T)
    if not mask.any(): return np.nan, np.nan
    diff = P[mask] - T[mask]
    rmse = float(np.sqrt(np.mean(diff**2)))
    mad = float(np.mean(np.abs(diff)))
    return rmse, mad

In [3]:
def _knn_predict_removed_with_S(X, XX, S, K=20, include_negative=False):
    X = np.asarray(X, dtype=float)
    XX = np.asarray(XX, dtype=float)
    S = np.asarray(S, dtype=float)
    n_items, n_users = XX.shape
    K_cap = min(int(K), max(0, n_users - 1))
    
    removed = (~np.isnan(X)) & (np.isnan(XX))
    if not np.any(removed): return {}, np.nan

    rated_mask = ~np.isnan(XX)
    user_means = np.nanmean(XX, axis=0)
    item_means = np.nanmean(XX, axis=1)
    global_mean = float(np.nanmean(XX)) if np.isfinite(np.nanmean(XX)) else 0.0

    preds = {}
    rows, cols = np.where(removed)
    
    for i, j in zip(rows, cols):
        cand = np.where(rated_mask[i])[0]
        cand = cand[cand != j]
        
        pred = global_mean
        if cand.size > 0 and K_cap > 0:
            sims = S[j, cand]
            if not include_negative:
                keep = np.where(sims > 0.0)[0]
                cand = cand[keep]
                sims = sims[keep]
            
            if cand.size > 0:
                K_eff = min(K_cap, cand.size)
                top = np.argsort(-sims)[:K_eff]
                nbrs = cand[top]
                w = sims[top]
                w_sum = np.sum(w)
                if np.isfinite(w_sum) and not np.isclose(w_sum, 0.0):
                    w = w / w_sum
                    pred = float(np.dot(w, XX[i, nbrs]))
                elif np.isfinite(user_means[j]): pred = float(user_means[j])
                elif np.isfinite(item_means[i]): pred = float(item_means[i])
            elif np.isfinite(user_means[j]): pred = float(user_means[j])
            elif np.isfinite(item_means[i]): pred = float(item_means[i])
        elif np.isfinite(user_means[j]): pred = float(user_means[j])
        elif np.isfinite(item_means[i]): pred = float(item_means[i])
            
        preds[(i, j)] = pred

    return preds, 0.0

## Main Process: Lambda Re-evaluation

In [22]:
for fold_num in FOLDS_TO_PROCESS:
    fold_id = f"fold_{fold_num:02d}"
    print(f"\n[{fold_id.upper()}] Loading Data...")
    
    fold_start = time.time()
    
    # Paths
    fold_data_dir = os.path.join(DATA_DIR, fold_id)
    fold_results_dir = os.path.join(RESULTS_DIR, fold_id)
    train_path = os.path.join(fold_data_dir, "train.csv")
    test_path = os.path.join(fold_data_dir, "test.csv")
    
    # 1. Load Data/History
    files = [f for f in os.listdir(fold_results_dir) 
             if f.startswith('alpha_optimization_history_') and f.endswith('.csv')]
    if not files: 
        print(f"  ⚠️  No alpha_optimization_history found. Skipping {fold_id}.")
        continue
    
    alpha_file = os.path.join(fold_results_dir, sorted(files)[-1])
    print(f"  Loading history: {os.path.basename(alpha_file)}")
    df_history = pd.read_csv(alpha_file)
    
    XX_train = _load_matrix_csv(train_path)
    XX_test = _load_matrix_csv(test_path)
    print(f"  Train: {XX_train.shape}, Test: {XX_test.shape}")


[FOLD_02] Loading Data...
  Loading history: alpha_optimization_history_20260121_022044.csv
  Train: (1682, 943), Test: (1682, 943)

[FOLD_03] Loading Data...
  Loading history: alpha_optimization_history_20260121_094002.csv
  Train: (1682, 943), Test: (1682, 943)

[FOLD_04] Loading Data...
  Loading history: alpha_optimization_history_20260121_184454.csv
  Train: (1682, 943), Test: (1682, 943)

[FOLD_05] Loading Data...
  Loading history: alpha_optimization_history_20260122_173925.csv
  Train: (1682, 943), Test: (1682, 943)


## Step 1: Select Optimal Alpha for Each Lambda

In [23]:
    # 2. Determine Optimal Alpha for ALL Lambdas first (to remove duplicates)
    print(f"  Selecting optimal alphas for {len(LAMBDA_VALUES)} lambda values...")
    
    lambda_selections = {}
    needed_evaluations = set() 
    
    for lam in LAMBDA_VALUES:
        df_history['new_penalty'] = lam * abs(df_history['alpha'] - 1.0)
        df_history['new_score'] = df_history['mse'] + df_history['new_penalty']
        
        # Best Alphas for this lambda
        best_idx = df_history.groupby(['method', 'K', 'TopN'])['new_score'].idxmin()
        best_df = df_history.loc[best_idx].copy()
        lambda_selections[lam] = best_df
        
        # Collect unique (Method, K, Alpha) combinations
        for _, row in best_df[['method', 'K', 'alpha']].iterrows():
            needed_evaluations.add((row['method'], row['K'], row['alpha']))
            
    tasks = sorted(list(needed_evaluations))
    print(f"  Optimization: Reduced {len(LAMBDA_VALUES) * len(tasks)} potential calls to {len(tasks)} unique predictions.")
    print(f"  Unique (method, K, alpha) combinations: {len(tasks)}")

  Selecting optimal alphas for 10 lambda values...
  Optimization: Reduced 8500 potential calls to 850 unique predictions.
  Unique (method, K, alpha) combinations: 850


## Step 2: Batch Prediction (Cache Results)

In [24]:
    # 3. Batch Evaluation (Predict once, cache results)
    evaluation_cache = {}  # Key: (Method, K, Alpha), Value: {rmse, mad}
    
    print(f"\n  Executing {len(tasks)} unique predictions...")
    loaded_sims = {}
    
    for i, (method, k, alpha) in enumerate(tasks):
        if i % 50 == 0 or i == len(tasks) - 1:
            print(f"    Progress: {i+1}/{len(tasks)} ({(i+1)/len(tasks)*100:.1f}%)")
        
        # Load Similarity Matrix
        if method not in loaded_sims:
            sim_path = os.path.join(fold_data_dir, f"sim_{method}.npy")
            if not os.path.exists(sim_path): 
                print(f"    ⚠️  Similarity file not found: {method}")
                continue
            loaded_sims[method] = np.load(sim_path)
        
        S_user = loaded_sims[method]
        S_alpha = _combine_single_similarity(S_user, alpha, clip_neg=True)
        
        # Predict on TEST data
        preds_dict, _ = _knn_predict_removed_with_S(
            X=XX_test.values, XX=XX_train.values, S=S_alpha, K=k
        )
        
        # Create Prediction DataFrame
        pred_df = XX_test.copy()
        pred_df[:] = np.nan
        for (item_idx, user_idx), r in preds_dict.items():
            pred_df.iloc[item_idx, user_idx] = r
            
        rmse, mad = rmse_mad_on_test(pred_df, XX_test)
        
        evaluation_cache[(method, k, alpha)] = {
            'rmse': rmse,
            'mad': mad
        }
        
    print(f"  ✅ Predictions complete!")


  Executing 850 unique predictions...
    Progress: 1/850 (0.1%)
    Progress: 51/850 (6.0%)
    Progress: 101/850 (11.9%)
    Progress: 151/850 (17.8%)
    Progress: 201/850 (23.6%)
    Progress: 251/850 (29.5%)
    Progress: 301/850 (35.4%)
    Progress: 351/850 (41.3%)
    Progress: 401/850 (47.2%)
    Progress: 451/850 (53.1%)
    Progress: 501/850 (58.9%)
    Progress: 551/850 (64.8%)
    Progress: 601/850 (70.7%)
    Progress: 651/850 (76.6%)
    Progress: 701/850 (82.5%)
    Progress: 751/850 (88.4%)
    Progress: 801/850 (94.2%)
    Progress: 850/850 (100.0%)
  ✅ Predictions complete!


## Step 3: Assemble Results with VALIDATION + TEST Metrics

In [25]:
    # 4. Assemble Results for each Lambda (V2: WITH VALIDATION METRICS)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    print(f"\n  Assembling results for {len(LAMBDA_VALUES)} lambda values...")
    
    for lam in LAMBDA_VALUES:
        best_df = lambda_selections[lam]
        results = []
        
        # Group by (Method, K, Alpha) to reuse cached predictions
        grouped = best_df.groupby(['method', 'K', 'alpha'])
        
        for (method, k, alpha), group in grouped:
            if (method, k, alpha) not in evaluation_cache: 
                continue
            
            cached = evaluation_cache[(method, k, alpha)]
            test_rmse = cached['rmse']
            test_mad = cached['mad']
            
            # V2: Extract VALIDATION metrics from history for each TopN
            for topn in group['TopN'].unique():
                # Find corresponding row in history
                hist_match = df_history[
                    (df_history['method'] == method) &
                    (df_history['K'] == k) &
                    (df_history['TopN'] == topn) &
                    (df_history['alpha'] == alpha)
                ]
                
                if hist_match.empty:
                    print(f"    ⚠️  Warning: No history for {method}, K={k}, TopN={topn}, α={alpha}")
                    continue
                
                hist_row = hist_match.iloc[0]
                reg_penalty = lam * abs(alpha - 1.0)
                reg_score = hist_row['mse'] + reg_penalty
                
                results.append({
                    'fold': fold_num,
                    'method': method,
                    'alpha': alpha,
                    'type': 'optimized',
                    'K': k,
                    'TopN': topn,
                    # V2: VALIDATION metrics (alpha selection basis)
                    'validation_mse': hist_row['mse'],
                    'validation_rmse': hist_row['rmse'],
                    'validation_precision': hist_row['precision'],
                    'validation_recall': hist_row['recall'],
                    'regularization_penalty': reg_penalty,
                    'regularized_score': reg_score,
                    # V2: TEST metrics (generalization evaluation)
                    'test_RMSE': test_rmse,
                    'test_MAD': test_mad,
                    'lambda': lam
                })
        
        # V2: Save with v2 suffix
        if results:
            output_path = os.path.join(fold_results_dir, f"grid_search_results_reeval_lambda_{lam}_v2_{timestamp}.csv")
            pd.DataFrame(results).to_csv(output_path, index=False)
            print(f"    ✓ Saved lambda={lam} ({len(results)} records)")

    print(f"\n  ✅ Fold {fold_num} Done in {time.time()-fold_start:.1f}s")


  Assembling results for 10 lambda values...
    ✓ Saved lambda=0.001 (1700 records)
    ✓ Saved lambda=0.002 (1700 records)
    ✓ Saved lambda=0.003 (1700 records)
    ✓ Saved lambda=0.004 (1700 records)
    ✓ Saved lambda=0.005 (1700 records)
    ✓ Saved lambda=0.01 (1700 records)
    ✓ Saved lambda=0.015 (1700 records)
    ✓ Saved lambda=0.02 (1700 records)
    ✓ Saved lambda=0.025 (1700 records)
    ✓ Saved lambda=0.03 (1700 records)

  ✅ Fold 5 Done in 587.1s


## Summary

In [26]:
print(f"\n========================================================")
print(f"ALL JOBS COMPLETE (V2)")
print(f"========================================================")
print(f"✅ Re-evaluation complete with VALIDATION + TEST metrics")
print(f"📁 Results saved with '_v2_' suffix")
print(f"\n📊 Output Schema:")
print(f"  - validation_mse, validation_rmse, validation_precision, validation_recall")
print(f"  - regularization_penalty, regularized_score")
print(f"  - test_RMSE, test_MAD")


ALL JOBS COMPLETE (V2)
✅ Re-evaluation complete with VALIDATION + TEST metrics
📁 Results saved with '_v2_' suffix

📊 Output Schema:
  - validation_mse, validation_rmse, validation_precision, validation_recall
  - regularization_penalty, regularized_score
  - test_RMSE, test_MAD


## Verification: Check Output Files

In [27]:
# Check generated files
import glob

for fold_num in FOLDS_TO_PROCESS:
    fold_id = f"fold_{fold_num:02d}"
    fold_results_dir = os.path.join(RESULTS_DIR, fold_id)
    
    v2_files = sorted(glob.glob(os.path.join(fold_results_dir, "grid_search_results_reeval_lambda_*_v2_*.csv")))
    
    print(f"\n{fold_id.upper()}: {len(v2_files)} v2 files generated")
    
    if v2_files:
        # Sample first file
        sample_df = pd.read_csv(v2_files[0])
        print(f"\nSample file: {os.path.basename(v2_files[0])}")
        print(f"Shape: {sample_df.shape}")
        print(f"\nColumns:")
        print(sample_df.columns.tolist())
        print(f"\nFirst 3 rows:")
        print(sample_df.head(3))


FOLD_02: 0 v2 files generated

FOLD_03: 0 v2 files generated

FOLD_04: 0 v2 files generated

FOLD_05: 20 v2 files generated

Sample file: grid_search_results_reeval_lambda_0.001_v2_20260127_173124.csv
Shape: (1700, 15)

Columns:
['fold', 'method', 'alpha', 'type', 'K', 'TopN', 'validation_mse', 'validation_rmse', 'validation_precision', 'validation_recall', 'regularization_penalty', 'regularized_score', 'test_RMSE', 'test_MAD', 'lambda']

First 3 rows:
   fold method  alpha       type   K  TopN  validation_mse  validation_rmse  \
0     5   acos   0.05  optimized  10     5        1.052747         1.026035   
1     5   acos   0.05  optimized  10    10        1.052747         1.026035   
2     5   acos   0.05  optimized  10    15        1.052747         1.026035   

   validation_precision  validation_recall  regularization_penalty  \
0              0.710976           0.563221                 0.00095   
1              0.661553           0.759052                 0.00095   
2              

In [ ]:
for fold_num in FOLDS_TO_PROCESS:
    fold_id = f"fold_{fold_num:02d}"
    print(f"\n[{fold_id.upper()}] Loading Data...")
    
    fold_start = time.time()
    
    # Paths
    fold_data_dir = os.path.join(DATA_DIR, fold_id)
    fold_results_dir = os.path.join(RESULTS_DIR, fold_id)
    train_path = os.path.join(fold_data_dir, "train.csv")
    test_path = os.path.join(fold_data_dir, "test.csv")
    
    # 1. Load Data/History
    files = [f for f in os.listdir(fold_results_dir) 
             if f.startswith('alpha_optimization_history_') and f.endswith('.csv')]
    if not files: 
        print(f"  ⚠️  No alpha_optimization_history found. Skipping {fold_id}.")
        continue
    
    alpha_file = os.path.join(fold_results_dir, sorted(files)[-1])
    print(f"  Loading history: {os.path.basename(alpha_file)}")
    df_history = pd.read_csv(alpha_file)
    
    XX_train = _load_matrix_csv(train_path)
    XX_test = _load_matrix_csv(test_path)
    print(f"  Train: {XX_train.shape}, Test: {XX_test.shape}")

    # 2. Determine Optimal Alpha for ALL Lambdas first (to remove duplicates)
    print(f"  Selecting optimal alphas for {len(LAMBDA_VALUES)} lambda values...")
    
    lambda_selections = {}
    needed_evaluations = set() 
    
    for lam in LAMBDA_VALUES:
        df_history['new_penalty'] = lam * abs(df_history['alpha'] - 1.0)
        df_history['new_score'] = df_history['mse'] + df_history['new_penalty']
        
        # Best Alphas for this lambda
        best_idx = df_history.groupby(['method', 'K', 'TopN'])['new_score'].idxmin()
        best_df = df_history.loc[best_idx].copy()
        lambda_selections[lam] = best_df
        
        # Collect unique (Method, K, Alpha) combinations
        for _, row in best_df[['method', 'K', 'alpha']].iterrows():
            needed_evaluations.add((row['method'], row['K'], row['alpha']))
            
    tasks = sorted(list(needed_evaluations))
    # Note: tasks calculation was using len(tasks) inside print which is fine now as it is defined above.
    # ORIGINAL CODE HAD BUG there? No, tasks = sorted(...) happens before print.
    print(f"  Optimization: Reduced {len(LAMBDA_VALUES) * df_history.groupby(['method', 'K', 'TopN']).ngroups if not df_history.empty else 0} potential (virtual) calls to {len(tasks)} unique predictions.") # Corrected print logic slightly to avoid confusion but okay to stick to original logic if it worked.
    # Original: print(f"  Optimization: Reduced {len(LAMBDA_VALUES) * len(tasks)} ...") <- This logic seems slightly circular/arbitrary in original but I'll stick to running the code.
    print(f"  Unique (method, K, alpha) combinations: {len(tasks)}")

    # 3. Batch Evaluation (Predict once, cache results)
    evaluation_cache = {}  # Key: (Method, K, Alpha), Value: {rmse, mad}
    
    print(f"\n  Executing {len(tasks)} unique predictions...")
    loaded_sims = {}
    
    for i, (method, k, alpha) in enumerate(tasks):
        if i % 50 == 0 or i == len(tasks) - 1:
            print(f"    Progress: {i+1}/{len(tasks)} ({(i+1)/len(tasks)*100:.1f}%)")
        
        # Load Similarity Matrix
        if method not in loaded_sims:
            sim_path = os.path.join(fold_data_dir, f"sim_{method}.npy")
            if not os.path.exists(sim_path): 
                print(f"    ⚠️  Similarity file not found: {method}")
                continue
            loaded_sims[method] = np.load(sim_path)
        
        S_user = loaded_sims[method]
        S_alpha = _combine_single_similarity(S_user, alpha, clip_neg=True)
        
        # Predict on TEST data
        preds_dict, _ = _knn_predict_removed_with_S(
            X=XX_test.values, XX=XX_train.values, S=S_alpha, K=k
        )
        
        # Create Prediction DataFrame
        pred_df = XX_test.copy()
        pred_df[:] = np.nan
        for (item_idx, user_idx), r in preds_dict.items():
            pred_df.iloc[item_idx, user_idx] = r
            
        rmse, mad = rmse_mad_on_test(pred_df, XX_test)
        
        evaluation_cache[(method, k, alpha)] = {
            'rmse': rmse,
            'mad': mad
        }
        
    print(f"  ✅ Predictions complete!")

    # 4. Assemble Results for each Lambda (V2: WITH VALIDATION METRICS)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    print(f"\n  Assembling results for {len(LAMBDA_VALUES)} lambda values...")
    
    for lam in LAMBDA_VALUES:
        best_df = lambda_selections[lam]
        results = []
        
        # Group by (Method, K, Alpha) to reuse cached predictions
        grouped = best_df.groupby(['method', 'K', 'alpha'])
        
        for (method, k, alpha), group in grouped:
            if (method, k, alpha) not in evaluation_cache: 
                continue
            
            cached = evaluation_cache[(method, k, alpha)]
            test_rmse = cached['rmse']
            test_mad = cached['mad']
            
            # V2: Extract VALIDATION metrics from history for each TopN
            for topn in group['TopN'].unique():
                # Find corresponding row in history
                hist_match = df_history[
                    (df_history['method'] == method) &
                    (df_history['K'] == k) &
                    (df_history['TopN'] == topn) &
                    (df_history['alpha'] == alpha)
                ]
                
                if hist_match.empty:
                    print(f"    ⚠️  Warning: No history for {method}, K={k}, TopN={topn}, α={alpha}")
                    continue
                
                hist_row = hist_match.iloc[0]
                reg_penalty = lam * abs(alpha - 1.0)
                reg_score = hist_row['mse'] + reg_penalty
                
                results.append({
                    'fold': fold_num,
                    'method': method,
                    'alpha': alpha,
                    'type': 'optimized',
                    'K': k,
                    'TopN': topn,
                    # V2: VALIDATION metrics (alpha selection basis)
                    'validation_mse': hist_row['mse'],
                    'validation_rmse': hist_row['rmse'],
                    'validation_precision': hist_row['precision'],
                    'validation_recall': hist_row['recall'],
                    'regularization_penalty': reg_penalty,
                    'regularized_score': reg_score,
                    # V2: TEST metrics (generalization evaluation)
                    'test_RMSE': test_rmse,
                    'test_MAD': test_mad,
                    'lambda': lam
                })
        
        # V2: Save with v2 suffix
        if results:
            output_path = os.path.join(fold_results_dir, f"grid_search_results_reeval_lambda_{lam}_v2_{timestamp}.csv")
            pd.DataFrame(results).to_csv(output_path, index=False)
            print(f"    ✓ Saved lambda={lam} ({len(results)} records)")

    print(f"\n  ✅ Fold {fold_num} Done in {time.time()-fold_start:.1f}s")


[FOLD_06] Loading Data...
  Loading history: alpha_optimization_history_20260128_053240.csv
  Train: (1682, 943), Test: (1682, 943)
  Selecting optimal alphas for 10 lambda values...
  Optimization: Reduced 17000 potential (virtual) calls to 940 unique predictions.
  Unique (method, K, alpha) combinations: 940

  Executing 940 unique predictions...
    Progress: 1/940 (0.1%)
    Progress: 51/940 (5.4%)
    Progress: 101/940 (10.7%)
    Progress: 151/940 (16.1%)
    Progress: 201/940 (21.4%)
    Progress: 251/940 (26.7%)
    Progress: 301/940 (32.0%)
    Progress: 351/940 (37.3%)
    Progress: 401/940 (42.7%)
    Progress: 451/940 (48.0%)
    Progress: 501/940 (53.3%)
    Progress: 551/940 (58.6%)
    Progress: 601/940 (63.9%)
    Progress: 651/940 (69.3%)
    Progress: 701/940 (74.6%)
    Progress: 751/940 (79.9%)
    Progress: 801/940 (85.2%)
    Progress: 851/940 (90.5%)
    Progress: 901/940 (95.9%)
    Progress: 940/940 (100.0%)
  ✅ Predictions complete!

  Assembling results for 